# Introduction

## 1. What is Collaborative Filtering?
Collaborative Filtering (CF) is a fundamental technique in recommender systems that automates the process of predicting a user's preference or rating for a specific item based on the observed preferences of a community of other users. Unlike content-based filtering, which relies on the intrinsic properties of items (e.g., genre, color, keywords), collaborative filtering relies solely on past user-item interactions. The foundational assumption is that if User A and User B have historically agreed on the quality of several items, they are likely to agree on a new, unseen item in the future.

## 2. Is it about User-to-User, Item-to-Item, or User-to-Item Similarity?
Collaborative Filtering encompasses **all three perspectives**, depending on the specific algorithmic approach adopted:
- **User-to-User Similarity (Memory-Based CF):** This approach directly compares the rating vectors of users. If the system needs to recommend an item to User $u$, it finds $k$ other users whose rating history is most similar to $u$'s (neighbors).
- **Item-to-Item Similarity (Memory-Based CF):** This approach transposes the matrix and compares items based on how users have rated them. If a user liked Item $i$, the system recommends other items that have the most similar rating profile to $i$. This was famously popularized by Amazon's "Customers Who Bought This Item Also Bought" feature.
- **User-to-Item Relationship (Model-Based CF / Matrix Factorization):** This approach does not compute explicit similarity between raw rows or columns. Instead, it models the interaction as an inner product of latent factors—a **User-to-Item** affinity. It discovers abstract features that explain the observed ratings, such that the predicted rating $\hat{r}_{ui}$ is a function of the user's preference vector and the item's characteristic vector.

## 3. When Exactly is the Proper Case and Time to Use Collaborative Filtering?
Collaborative Filtering is the appropriate methodology under the following specific conditions:
- **Sparse User Feedback:** You have a large user base and a large item catalog, but each individual user has only interacted with a tiny fraction of the total items.
- **Subjective or Aesthetic Domains:** The items lack objective, machine-readable attributes (e.g., movies, music, jokes, handmade crafts). For example, you cannot analyze the "pixel data" of a movie to determine if it is "funny"; you must rely on what other people who laughed at the same things thought.
- **Cross-Category Discovery:** You want to surprise the user with recommendations from categories they have never visited (serendipity). For instance, a user who likes *Blade Runner* might like a specific Philip K. Dick novel, an association only visible through overlapping fanbases, not through text analysis.

## 4. Where Can We Implement Collaborative Filtering?
Collaborative Filtering is implemented across a wide spectrum of digital services where user decision fatigue is high:
- **E-commerce Platforms:** Amazon, eBay, and Etsy use it for product recommendations on detail pages and checkout cross-sells.
- **Streaming Media Services:** Netflix (movie suggestions), Spotify (Discover Weekly playlist), and YouTube (video recommendations) rely heavily on CF to keep users engaged by surfacing deep catalog content.
- **Social Networks:** LinkedIn ("People You May Know"), Facebook (Friend Suggestions), and Twitter (Who to Follow) treat "following a user" as the item interaction.
- **Digital Advertising:** Ad exchanges use CF to predict Click-Through Rate (CTR) for a specific user-ad pair based on similar user cohorts.

## 5. Why Should We Use Collaborative Filtering?
The primary justification for using Collaborative Filtering is **domain independence and discovery**. We should use it because:
1.  **No Feature Engineering Required:** It eliminates the need for subject matter experts to manually tag every song with "distorted guitar" or every book with "unreliable narrator." The system learns these latent descriptors from the data itself.
2.  **Quality of Insight:** It captures nuanced, hard-to-quantify preferences (e.g., "Movies with a bittersweet ending").
3.  **Scalability of Model Training:** Modern matrix factorization algorithms are highly parallelizable and can handle matrices with hundreds of millions of rows and columns efficiently using Stochastic Gradient Descent (SGD) or Alternating Least Squares (ALS).

## 6. Step-by-Step Mathematical Workflow with Formulation
When we have the data—specifically a set of triplets $(u, i, r_{ui})$ where $u$ is user ID, $i$ is item ID, and $r_{ui}$ is the interaction strength (explicit rating or implicit count)—we work with **Model-Based Collaborative Filtering (Matrix Factorization)** as follows:

### **Step 1: Define the Interaction Matrix $R$**
We construct a sparse matrix $R \in \mathbb{R}^{m \times n}$, where $m$ is the number of users and $n$ is the number of items. Most entries $r_{ui}$ are missing (NaN).

### **Step 2: Mathematical Model Definition (Hypothesis)**
We assume each user $u$ can be represented by a latent vector $p_u \in \mathbb{R}^k$ and each item $i$ by a latent vector $q_i \in \mathbb{R}^k$. Here, $k$ is the number of latent factors (dimensionality of the abstract space, e.g., $k=50$ or $100$).
The predicted rating $\hat{r}_{ui}$ is modeled as the dot product (interaction) between the user and item vectors:
$$
\hat{r}_{ui} = q_i^T p_u = \sum_{f=1}^{k} q_{if} \cdot p_{uf}
$$

### **Step 3: Define the Loss Function (Objective)**
We need to minimize the difference between the observed ratings $r_{ui}$ and the predicted ratings $\hat{r}_{ui}$. To prevent overfitting to the sparse data, we introduce **L2 Regularization**. The objective function $\mathcal{L}$ to minimize is:
$$
\mathcal{L} = \min_{p^*, q^*} \sum_{(u,i) \in \mathcal{K}} \left( r_{ui} - q_i^T p_u \right)^2 + \lambda \left( \|q_i\|^2 + \|p_u\|^2 \right)
$$

Where:
- $\mathcal{K}$ is the set of known (user, item) pairs in the training data.
- $\lambda$ is the regularization hyperparameter controlling the penalty on vector magnitude.

### **Step 4: Optimization via Stochastic Gradient Descent (SGD)**
We iterate over each known rating $r_{ui}$ in the training set. We compute the prediction error:
$$
e_{ui} = r_{ui} - q_i^T p_u
$$

We then compute the gradients of the loss with respect to the parameters and update them in the opposite direction of the gradient. **Update for User Vector $p_u$:**
$$
p_u \leftarrow p_u + \eta \cdot (e_{ui} \cdot q_i - \lambda \cdot p_u)
$$

**Update for Item Vector $q_i$:**
$$
q_i \leftarrow q_i + \eta \cdot (e_{ui} \cdot p_u - \lambda \cdot q_i)
$$

Where $\eta$ is the learning rate.

### **Step 5: Prediction for Missing Values**
After convergence (or a fixed number of epochs), we reconstruct the full matrix $\hat{R}$. For a user $u$ and an unseen item $j$, the final predicted score is:
$$
\hat{r}_{uj} = q_j^T p_u
$$
We then recommend the top-$N$ items with the highest $\hat{r}_{uj}$ values that the user has not yet interacted with.

# Module Loading

In [3]:
import gdown
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pylab import rcParams
import pandas as pd
import numpy as np
from google.colab import drive
from warnings import filterwarnings
import duckdb
import pandas as pd
from typing import Optional, Union, List, Dict, Any, ContextManager
from pathlib import Path
from contextlib import contextmanager

filterwarnings('ignore')

drive.mount('/content/drive', force_remount = False)
dbpath = '/content/drive/MyDrive/Colab Notebooks/DB_ga4_ecommerce.duckdb'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Modern Professional Color Palette
modern_colors = [
    "#1f77b4",   # Vibrant Blue (Primary)
    "#ff7f0e",   # Bright Orange (Accent/Comparison)
    "#2ca02c",   # Fresh Green (Success/Positive)
    "#d62728",   # Soft Red (Alert/Warning)
    "#9467bd",   # Elegant Purple
    "#8c564b",   # Warm Brown
    "#e377c2",   # Pink
    "#7f7f7f",   # Neutral Gray
    "#bcbd22",   # Olive/Yellow-Green
    "#17becf"    # Cyan/Teal
]

# 2. Main Style Dictionary
modern_light_style = {
    # Background - Clean and bright
    "figure.facecolor": "#ffffff",
    "axes.facecolor": "#f8f9fa",
    "savefig.facecolor": "#ffffff",

    # Grid - Very subtle and non-distracting
    "axes.grid": True,
    "grid.color": "#e6e8eb",
    "grid.linestyle": "--",
    "grid.linewidth": 0.8,
    "axes.grid.which": "both",

    # Typography
    "text.color": "#1f2937",
    "axes.labelcolor": "#1f2937",
    "xtick.color": "#374151",
    "ytick.color": "#374151",
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.titlepad": 18,
    "font.size": 12,
    "font.family": "sans-serif", # You can change to 'Arial', 'Helvetica', etc.

    # Spines - Clean and minimal
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": True,
    "axes.spines.bottom": True,
    "axes.edgecolor": "#4b5563",
    "axes.linewidth": 1.2,

    # Lines and markers
    "axes.prop_cycle": plt.cycler(color=modern_colors),
    "lines.linewidth": 2.5,
    "lines.markersize": 7,
    "lines.markeredgewidth": 0.8,

    # Patches (bars, areas, etc.)
    "patch.edgecolor": "#ffffff",
    "patch.linewidth": 0.8,

    # Legend
    "legend.frameon": False,
    "legend.loc": "best",
    "legend.fontsize": 11,
}

plt.style.use('default')
sns.set_theme(style="whitegrid", rc=modern_light_style)
plt.rcParams.update(modern_light_style)
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=modern_colors)

In [5]:
class DuckDBManager:
    """
    Manager class for DuckDB connections and queries.

    Example:
        db = DuckDBManager('mydb.duckdb')
        df = db.query("SELECT * FROM users")
        db.close()

        # Or use context manager:
        with DuckDBManager('mydb.duckdb') as db:
            df = db.query("SELECT * FROM users")
    """

    def __init__(self,
                 db_path: Union[str, Path],
                 read_only: bool = True,
                 threads: Optional[int] = None,
                 memory_limit: Optional[str] = '2GB'):
        """
        Initialize DuckDB manager.

        Args:
            db_path: Path to database file (use ':memory:' for in-memory DB)
            read_only: Open in read-only mode
            threads: Number of CPU threads to use
            memory_limit: Memory limit (e.g., '4GB', '1TB')
        """
        self.db_path = str(db_path)
        self.read_only = read_only
        self.conn = None

        # Create connection
        self._create_connection(threads, memory_limit)

    def _create_connection(self, threads: Optional[int] = None,
                          memory_limit: Optional[str] = None):
        """Create DuckDB connection with optional settings."""
        self.conn = duckdb.connect(database=self.db_path, read_only=self.read_only)

        # Configure settings
        if threads:
            self.conn.execute(f"SET threads = {threads}")
        if memory_limit:
            self.conn.execute(f"SET memory_limit = '{memory_limit}'")

    def query(self,
              query: str,
              params: Optional[Union[List, Dict, tuple]] = None,
              fetch_size: Optional[int] = None) -> pd.DataFrame:
        """
        Execute query and return DataFrame.

        Args:
            query: SQL query string
            params: Query parameters
            fetch_size: Number of rows to fetch (None for all)

        Returns:
            pandas DataFrame
        """
        if self.conn is None:
            raise ValueError("Connection is closed. Please reconnect.")

        try:
            if params is not None:
                result = self.conn.execute(query, params)
            else:
                result = self.conn.execute(query)

            if fetch_size:
                return result.fetch_df_chunk(fetch_size)
            return result.fetchdf()

        except Exception as e:
            raise Exception(f"Query failed: {e}\nQuery: {query}")

    def query_arrow(self, query: str, params: Optional[Union[List, Dict]] = None):
        """Execute query and return Arrow table (faster for large datasets)."""
        if params is not None:
            return self.conn.execute(query, params).fetch_arrow_table()
        return self.conn.execute(query).fetch_arrow_table()

    def register_dataframe(self, name: str, df: pd.DataFrame):
        """Register pandas DataFrame as temporary table (table_view)."""
        self.conn.register(name, df)

    def ListedTable(self):
        tables = self.conn.execute("SELECT table_name FROM duckdb_tables()").df()
        data = list()
        for table in tables['table_name']:
            count = self.conn.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
            data.append({'table_name': table, 'row_count': count})
        info_df = pd.DataFrame(data)
        info_df = info_df.sort_values(by='row_count', ascending=False)
        info_df['row_count'] = info_df['row_count'].apply(lambda x: f"{x:,}")
        display(info_df)
        return info_df

    def execute(self, query: str, params: Optional[Union[List, Dict]] = None):
        """Execute query without returning results."""
        if params is not None:
            self.conn.execute(query, params)
        else:
            self.conn.execute(query)

    def table_exists(self, table_name: str) -> bool:
        """Check if table exists in database."""
        result = self.query(
            "SELECT COUNT(*) FROM information_schema.tables WHERE table_name = ?",
            [table_name]
        )
        return result.iloc[0, 0] > 0

    def get_tables(self) -> List[str]:
        """Get list of all tables in database."""
        df = self.query("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'")
        return df['table_name'].tolist()

    def get_schema(self, table_name: str) -> pd.DataFrame:
        """Get schema information for a table."""
        return self.query(f"DESCRIBE {table_name}")

    def close(self):
        """Close database connection."""
        if self.conn:
            self.conn.close()
            self.conn = None

    def __enter__(self):
        """Context manager entry."""
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Context manager exit."""
        self.close()


In [6]:
@contextmanager
def duckdb_connection(db_path: Union[str, Path],
                      read_only: bool = False,
                      **kwargs) -> DuckDBManager:
    """
    Context manager for DuckDB connection.

    Example:
        with duckdb_connection('mydb.duckdb') as db:
            df = db.query("SELECT * FROM users")
    """
    db = DuckDBManager(db_path, read_only, **kwargs)
    try:
        yield db
    finally:
        db.close()

# Data Loading

In [7]:
db = DuckDBManager(db_path = dbpath)
_ = db.ListedTable()

mas_train = db.query('SELECT * FROM FinalMasterData')

,table_name,row_count
0,FinalMasterData,"1,014,426"
13,MasterTrainData,"1,014,426"
2,FinalMasterData_train,"811,540"
1,FinalMasterData_test,"202,886"
14,UserFeature_Analysis,"47,474"
3,fpgrowth_transactions,"46,375"
12,mapping_product_name,429
9,mapping_primary_country,109
7,mapping_department_id,81
5,mapping_brand,7


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
import json

predpath = os.path.dirname(dbpath)
print(predpath)
#with open(json_filename, 'r') as f:
#    loaded_data = json.load(f)

/content/drive/MyDrive/Colab Notebooks


In [13]:
predpath = os.path.dirname(dbpath)
if 'predpath' in locals() or 'predpath' in globals():
    print(f"Searching for ZIP files in: {predpath}")
    zip_files = [f for f in os.listdir(predpath) if f.endswith('.zip')]

    if zip_files:
        print("Found zip files:")
        for jf in zip_files:
            print(os.path.join(predpath, jf))
    else:
        print("No zip files found in this directory.")
else:
    print("Error: 'predpath' variable is not defined. Please ensure the previous cell setting predpath was executed.")

Searching for ZIP files in: /content/drive/MyDrive/Colab Notebooks
Found zip files:
/content/drive/MyDrive/Colab Notebooks/20260418_fpgrowth_prediction.zip


In [33]:
from shutil import copy
from zipfile import ZipFile

destination_dir = '/content/'
all_extracted_json_data = list()

print(f"Processing ZIP files from {predpath}:")
for zip_file_name in zip_files:
    src_zip_path = os.path.join(predpath, zip_file_name)
    dest_zip_path = os.path.join(destination_dir, zip_file_name)

    if not os.path.exists(src_zip_path):
        print(f"Warning: Source ZIP file not found: {src_zip_path}. Skipping.")
        continue
    copy(src_zip_path, dest_zip_path)
    with ZipFile(dest_zip_path, 'r') as zip_ref:
        zip_ref.extractall(destination_dir)

Processing ZIP files from /content/drive/MyDrive/Colab Notebooks:
Copying '20260418_fpgrowth_prediction.zip' to '/content/'...
Copy complete.
Extracting '/content/20260418_fpgrowth_prediction.zip'...
Extraction complete.


In [34]:
import json
jsonfile = [f for f in os.listdir(destination_dir) if f.endswith('.json')]
choosejson = jsonfile[-1]

with open(choosejson, 'r') as f:
    fpgrowthdata = json.load(f)

In [31]:
from itertools import islice

def dictslice(jsonfile:dict,
              count : int = 10,
              show : bool = True) -> dict:
    Data = dict(islice(jsonfile.items(), int(count)))
    if show:
        print(Data)
    return Data

a = dictslice(fpgrowthdata, 3)

{'1000684.125': [{'items': ['9196908'], 'support': 14}, {'items': ['9199084'], 'support': 44}], '1001326.125': [{'items': ['9197398'], 'support': 34}, {'items': ['9197946'], 'support': 10}], '1010983.25': [{'items': ['9196832'], 'support': 25}]}


In [28]:
def Json2Dataframe(jsondata : dict) -> pd.DataFrame:
    records = ({"group_id": group_id,
                "items"   : entry["items"],
                "support" : entry["support"]}
        for group_id, entries in jsondata.items()
        for entry in entries)
    data = pd.DataFrame.from_records(records)
    data["items"] = data["items"].apply(
        lambda x: ", ".join(x) if isinstance(x, (list, set)) else x)
    return data

DataFPGpredict = Json2Dataframe(fpgrowthdata)
display(DataFPGpredict.describe(include='all'))

,group_id,items,support
count,27290,27290,27290.000000
unique,10631,677,NaN
top,59061040.0,9188192,NaN
freq,10,1958,NaN
mean,NaN,NaN,48.772774
std,NaN,NaN,37.048558
min,NaN,NaN,8.000000
25%,NaN,NaN,19.000000
50%,NaN,NaN,38.000000
75%,NaN,NaN,72.000000


In [27]:
df.shape

(27290, 3)